# Gemini API

Gemini API from/to translation helpers

In [ ]:
#| default_exp gemini

## Imports

In [ ]:
#| export
import json
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fastspec.errors import api_error_from_event

from fastllm.types import *
from fastllm.streaming import *
from fastllm.streaming import _acollect_stream

In [ ]:
#| hide
import yaml,base64,httpx
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from toolslm.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

In [ ]:
specs_path = files('fastllm') / 'specs'
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))
gem_spec

SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79)

In [ ]:
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

## Normalize

Helpers to transform API responses into the internal representation types

### Tool Calls

Normalize tool calls shape(s)

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

As a general rule, if you receive a thought signature in a model response, you should pass it back exactly as received when sending the conversation history in the next turn. When using Gemini 3 models, you must pass back thought signatures during function calling, otherwise you will get a validation error (4xx status code). This includes when using the minimal thinking level setting for Gemini 3 Flash.

from: https://ai.google.dev/gemini-api/docs/thought-signatures

In [ ]:
#| export
def norm_tool_calls(resp):
    "Extract Gemini functionCall parts as normalized tool calls."
    out = []
    for i,p in enumerate(nested_idx(resp, 'candidates', 0, 'content', 'parts') or []):
        if not (fc:=p.get("functionCall")): continue
        extra = {k:v for k,v in p.items() if k != "functionCall"}
        out.append(ToolCall(id=fc.get("id", f"call_{i}"), name=fc.get("name", ""), arguments=fc.get("args") or {}, extra=extra))
    return out

User defined:

In [ ]:
mn = 'models/gemini-3-flash-preview'

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "What is 3 + 5 and 10+ 20? Use the simple_add tool in parallel."}]}], tools=[{"functionDeclarations": [sch['function']]}])
norm_tool_calls(resp)

[ToolCall(id='mqa834kq', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'thoughtSignature': 'EtICCs8CAQw51sc6mgqlDQrSy8PGRpqT/CAa8k3y91U17/o5lB2M0r1CT31GhWiL/ZDg6uNptErERw3Tc2MBdzgG2Y9UE/RdE02sz3U6hwIUiduazNMTTNOyscJoxiH53aenOTvxBinUYFFO0Yu8av1FOSPW3FmIbmGyp22wmrkzFE7W31GZUYV2K89jtKJI4XJkP7WrDBknCz4td7sXx9IyjAf3PJVDIWBdnCkM/1OzxBgEAL5yY3o/f0gPHewo8LsBCZ/KRK2F/WQh1Bq1rn2HsSyuBCqVhxzbzc+P/xAgfhG1zSzqbFp75RIlUpvMB8zIR2wNIu2aZJ5NgNmEuPZcNghRGpd3Dvla5+LCZpvbHvNGNccA6SZyNp5KzMCOT+yHMtiQuhHTDTPxRMyqY+a79V3uGGTu2e6AvA8pqXUca4yCYWE8s1dVyCUmlQcZGDGaURs='}),
 ToolCall(id='ba3gimpc', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={})]

In [ ]:
model_parts = [{"functionCall": {"name": tc.name, "args": tc.arguments}, **tc.extra} for tc in  norm_tool_calls(resp)]
func_results = [
    {"functionResponse": {"name": "simple_add", "response": {"result": 8}}},
    {"functionResponse": {"name": "simple_add", "response": {"result": 30}}},
]
resp_gem = await gem_cli.models.generate_content(
    model=mn,
    contents=[
        {"role": "user",  "parts": [{"text": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]},
        {"role": "model", "parts": model_parts},
        {"role": "user",  "parts": func_results},
    ],
    tools=[{"functionDeclarations": [sch['function']]}]
)
resp_gem

{'candidates': [{'content': {'parts': [{'text': 'OK. 3 + 5 is 8, and 10 + 20 is 30.'}],
    'role': 'model'},
   'finishReason': 'STOP',
   'index': 0}],
 'usageMetadata': {'promptTokenCount': 232,
  'candidatesTokenCount': 24,
  'totalTokenCount': 256,
  'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 232}]},
 'modelVersion': 'gemini-3-flash-preview',
 'responseId': 'Rkfvace8G6LDvdIPipDeuQ8'}

Gemini server tool calls aren't returned as tools, however details can be found in `groundingMetadata`:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}], tools=[{"googleSearch": {}}])
norm_tool_calls(resp)

[]

In [ ]:
str(nested_idx(resp, 'candidates', 0, 'groundingMetadata'))[:100]

"{'searchEntryPoint': {'renderedContent': '<style>\\n.container {\\n  align-items: center;\\n  border-ra"

Code execution:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using the code execution tool"}]}], tools=[{"codeExecution": {}}]
)
norm_tool_calls(resp)

[]

In [ ]:
resp

{'candidates': [{'content': {'parts': [{'executableCode': {'language': 'PYTHON',
       'code': 'def fibonacci(n):\n    fib_sequence = [0, 1]\n    while len(fib_sequence) < n:\n        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])\n    return fib_sequence[:n]\n\nprint(fibonacci(10))',
       'id': '8twgmktq'},
      'thoughtSignature': 'EsECCr4CAQw51sfS32o/WlF0pq9Xd+00wPLkhEXwsgUloO/O6aiSMmf/Ssq5ywAo4TeG5tUMCN1pEoj/6WlUfGkQubEaD5VL2u/wEFUHwtB7Pszvb61vY/xYgyaFR9JdRZOk0ERRCeRQfFUmMjL7pWpAqfj/TrQaWaFQU28cd8MRqGDYmyraxQ6q8vdCaaLZKpX2kMSyk1Nn5oEl6lW0eHdR+zdMm49ejiVXU4IWg+lYE6zfBQXBj00V1AFEHLU5NKFh4cnFcxOvnH+09vpEk04gxM2R+wkFY7eou7lFd5qMeO6J3bseLiucX/IPDqoBnz+Bfgpx4NPBbrW/JURs9ZDzbQg/6NelDvy1EgTQu1AsN3eY00O0x4isOhj4iVjudcZMVmdIcMvo9wXX50G2pAOiUhHId++6m4A6e2VZz3OJTMrY'},
     {'codeExecutionResult': {'outcome': 'OUTCOME_OK',
       'output': '[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\n',
       'id': '8twgmktq'}},
     {'text': 'The first 10 Fibonacci numbers are: 0, 1, 1, 2, 3, 5, 8, 13, 

In [ ]:
str(nested_idx(resp, 'candidates', 0, 'content', 'parts', 0))[:100]

"{'executableCode': {'language': 'PYTHON', 'code': 'def fibonacci(n):\\n    fib_sequence = [0, 1]\\n   "

In [ ]:
str(nested_idx(resp, 'candidates', 0, 'content', 'parts', 1))

"{'codeExecutionResult': {'outcome': 'OUTCOME_OK', 'output': '[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]\\n', 'id': '8twgmktq'}}"

Computer Use is returned as a regular tool and expects a result:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "Go to google.com and search for 'hello world'"}]}], tools=[{"computerUse": {"environment": "ENVIRONMENT_BROWSER"}}])
norm_tool_calls(resp)

[ToolCall(id='otbkk94n', name='open_web_browser', arguments={}, server=False, extra={'thoughtSignature': 'Ev8BCvwBAQw51sec6jqeeIi0Tt8UKqnO6gxOcwO/YLz6xpCezoC/shUvP9kBg4Gd4NjsqAOioYL7h7mCSRrdBlAg0X/WjtmFN5XxTlXOwcCz+jseXbKlYq3vMU46xDq/cnlQ/c+SYl9Lhc1kpxN1zlpyWJtmv3BWP30r5g+CcmxfkRIrtTmNepPdlEBegN5I+PU3pQ4AcJRQVFOkZjwj23vcF04rJSdRZ3siFSv3Y+kGSs0PmLGC2niVMuZ3Dum607KtxSvwBYNDhdIQ6j6RXKlsyPZfSLtEK1vVGf2jGdRBF133ORkEAILL3HNjcD1kQbqfqn3j8AZdVYxxbSpfYlJq'})]

### Usage

Normalize usage shape(s)

In [ ]:
#| export
def norm_usage(resp):
    "Normalize Gemini usageMetadata shape."
    if not (usg:=resp.get("usageMetadata")): return None
    pt = int(usg.get("promptTokenCount", 0) or 0)
    ct = int(usg.get("candidatesTokenCount", 0) or 0)
    tt = int(usg.get("totalTokenCount", pt + ct) or (pt + ct))
    cached = int(usg.get("cachedContentTokenCount", 0) or 0)
    reasoning = int(usg.get("thoughtsTokenCount", 0) or 0)
    parts = nested_idx(resp, 'candidates', 0, 'content', 'parts') or []
    cand = nested_idx(resp, 'candidates', 0) or {}
    stu = {}
    if any("executableCode" in p for p in parts): stu["code_execution"] = sum(1 for p in parts if "executableCode" in p)
    if "groundingMetadata" in cand: stu["google_search"] = 1
    if stu: usg['server_tool_use'] = stu
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=tt,
                 cached_tokens=cached, reasoning_tokens=reasoning, raw=usg)

Text response:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "hi!"}]}])
norm_usage(resp)

Usage(prompt_tokens=3, completion_tokens=9, total_tokens=78, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=66, raw={'promptTokenCount': 3, 'candidatesTokenCount': 9, 'totalTokenCount': 78, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 66})

Server tools:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}], tools=[{"googleSearch": {}}])
norm_usage(resp)

Usage(prompt_tokens=73, completion_tokens=213, total_tokens=941, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=655, raw={'promptTokenCount': 73, 'candidatesTokenCount': 213, 'totalTokenCount': 941, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 73}], 'thoughtsTokenCount': 655, 'server_tool_use': {'google_search': 1}})

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code"}]}], tools=[{"codeExecution": {}}])
norm_usage(resp)

Usage(prompt_tokens=12, completion_tokens=261, total_tokens=557, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=284, raw={'promptTokenCount': 12, 'candidatesTokenCount': 261, 'totalTokenCount': 557, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 12}], 'thoughtsTokenCount': 284})

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "What is solveit? https://solve.it.com/"}]}], tools=[{"urlContext": {}}])
norm_usage(resp)

Usage(prompt_tokens=101, completion_tokens=472, total_tokens=4881, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=1040, raw={'promptTokenCount': 101, 'candidatesTokenCount': 472, 'totalTokenCount': 4881, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 101}], 'toolUsePromptTokenCount': 3268, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3268}], 'thoughtsTokenCount': 1040, 'server_tool_use': {'google_search': 1}})

In [ ]:
str(nested_idx(resp, 'candidates', 0, 'content', 'parts', 0, 'text'))[:300]

'**SolveIt** is a problem-solving methodology and a software platform created by **Jeremy Howard** (founder of fast.ai) and **Eric Ries** (author of *The Lean Startup*). It is a product of their company, Answer.AI.\n\nThe project is designed to change how people use AI for coding, writing, and learning'

### Finish Reason

Normalize finish reason shape(s)

In [ ]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    reason = nested_idx(resp, 'candidates', 0, 'finishReason')
    if reason is not None: reason = reason.lower()
    mp = dict(stop=FinishReason.stop, max_tokens=FinishReason.length, safety=FinishReason.content_filter, blocklist=FinishReason.content_filter)
    r =  mp.get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [ ]:
norm_finish(resp)

<finish_reason.stop: 'stop'>

### Non-stream Completion

Normalize response into Completion.

In [ ]:
#| export
def _gem_part_type(p):
    "Map Gemini part to canonical PartType."
    if 'functionCall' in p or 'toolCall' in p: return PartType.tool_use
    if 'executableCode' in p or 'codeExecutionResult' in p: return PartType.tool_result
    if p.get('thought'): return PartType.thinking
    return PartType.text

def norm_parts(resp):
    "Normalize Gemini generateContent response."
    c0 = nested_idx(resp, 'candidates', 0) or {}
    tcs = norm_tool_calls(resp)
    tc_map = {tc.id: tc for tc in tcs}
    parts = []
    for p in nested_idx(c0, 'content', 'parts') or []:
        typ = _gem_part_type(p)
        if typ == 'tool_use':
            fc = p.get('functionCall') or p.get('toolCall') or {}
            tc = tc_map.get(fc.get('id'))
            if tc: 
                tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
                parts.append(Part(type=PartType.tool_use, data=tdata))
        else: parts.append(Part(type=typ, text=p.get("text",""), data=p))
    if citations := c0.get('groundingMetadata'):
        for p in parts: 
            if p.type == PartType.text: p.data['citations'] = citations
    return parts

Users can define `norm_tool_calls`, `norm_parts`, `norm_finish` and `norm_usage` for a given provider, and `mk_completion` will be used to create the canonical `Completion` object based on those functions:

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage)
api_registry.register('gemini', **norm_funcs)

#### Text

In [ ]:
api_name,vnd_nm = 'gemini', 'gemini'

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "Say hi in one word"}]}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hi

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=6, completion_tokens=1, total_tokens=76, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=69, raw={'promptTokenCount': 6, 'candidatesTokenCount': 1, 'totalTokenCount': 76, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 69})`

</details>

#### Thinking (mocked)

In [ ]:
comp = mk_completion({"candidates": [{"content": {"parts": [{"text": "Let me think...", "thought": True}, {"text": "Here's the answer."}], "role": "model"}, "finishReason": "STOP"}], "usageMetadata": {"promptTokenCount": 10, "candidatesTokenCount": 20, "totalTokenCount": 30}}, mn, api_name, vnd_nm)
comp

<details><summary>Thinking</summary>

Let me think...

</details>

Here's the answer.

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=20, total_tokens=30, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'promptTokenCount': 10, 'candidatesTokenCount': 20, 'totalTokenCount': 30})`

</details>

#### Tool call

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "What is 3 + 5? Use simple_add tool."}]}], tools=[{"functionDeclarations": [sch['function']]}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'b': 5, 'a': 3})


<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=66, completion_tokens=18, total_tokens=134, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=50, raw={'promptTokenCount': 66, 'candidatesTokenCount': 18, 'totalTokenCount': 134, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 66}], 'thoughtsTokenCount': 50})`

</details>

In [ ]:
test_eq(comp.tool_calls[0].server, False)

Web search response:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}], tools=[{"googleSearch": {}}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Today, Friday, April 24, 2026, the weather in Istanbul is primarily **sunny** and mild. 

Here are the key details for today's forecast:
*   **Temperature:** Current temperature is approximately **16°C (60°F)**. The daytime high is expected to reach **16°C (61°F)**, while the low tonight will drop to around **10°C (50°F)**.
*   **Conditions:** It is sunny throughout the day, with a few clouds potentially appearing later in the evening.
*   **Precipitation:** There is a very low chance of rain (around 10% during the day and 0% currently).
*   **Humidity:** Humidity levels are around **56% to 63%**, making for a relatively comfortable day.

If you are planning to be outdoors, it is a great day for it, though you might want a light jacket for the cooler evening hours.

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=73, completion_tokens=213, total_tokens=941, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=655, raw={'promptTokenCount': 73, 'candidatesTokenCount': 213, 'totalTokenCount': 941, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 73}], 'thoughtsTokenCount': 655, 'server_tool_use': {'google_search': 1}})`

</details>

In [ ]:
test_eq(comp.usage.raw['server_tool_use']['google_search'], 1)

Code Execution:

In [ ]:
resp = await gem_cli.models.generate_content(model=mn, contents=[{"role": "user", "parts": [{"text": "Calculate the first 10 fibonacci numbers using code execution tool"}]}], tools=[{"codeExecution": {}}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

The first 10 Fibonacci numbers are: 0, 1, 1, 2, 3, 5, 8, 13, 21, and 34.

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=111, completion_tokens=43, total_tokens=542, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'promptTokenCount': 111, 'candidatesTokenCount': 43, 'totalTokenCount': 542, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 111}], 'toolUsePromptTokenCount': 388, 'toolUsePromptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 388}], 'server_tool_use': {'code_execution': 1}})`

</details>

In [ ]:
test_eq(comp.usage.raw['server_tool_use']['code_execution'], 1)

### Streaming

Streaming module implements `PartAccum` and `_acollect_stream` which are used to collate canonical `Delta` objects into a final `Completion` object similar to the non-streaming path while yielding text/thinking/citations. Each provider needs to implement `norm_sse_event` which will transform an event into a compatible `Delta` object which will be consumed by the streaming module

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o,postproc=noop):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            if citations:= o.get('citations'): print(postproc(citations),end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

OpenAI responses api returns the whole tool call response with a `response.output_item.done` event which doesn't require argument collation and makes our job easy. If api wasn't returning the collated tool call, we'd have needed to provide delta arguments in Delta object's `tool_call` like:

```py
ToolCall(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, extra=extra)
```

In [ ]:
#| export
def norm_sse_event(ev, **kwargs):
    "Normalize Gemini stream event into Delta."
    cand = nested_idx(ev, 'candidates', 0) or {}
    finish_reason = norm_finish(ev)
    parts = nested_idx(cand, 'content', 'parts') or []
    thinking = "".join(p.get("text","") for p in parts if p.get("thought") and "text" in p)
    txt = "".join(p.get("text","") for p in parts if not p.get("thought") and "text" in p)
    tcs = norm_tool_calls(ev)
    if ev.get("error"): raise api_error_from_event(ev)
    return Delta(text=txt, thinking=thinking, tool_calls=tcs, citations=listify(cand.get('groundingMetadata', [])), finish_reason=finish_reason, usage=norm_usage(ev), raw=ev, **kwargs)

Each provider also needs to implement `delta_index_fn` which returns a current idx and last idx and used by `PartAccum` like `part_accum.append(typ, idx, **tc_kwargs)`.

Here `d` is a `Delta` object, `typ` is the name of the event, it also takes last event's name and it's index to decide how to do collation

Gemini returns collated tool calls instead of streaming in parts and texts are not chunked into very small pieces. Also, it doesn't have group index like Anthropic so we simply increment the each delta index by 1, and let PartAccum handle text and thinking collation.

In [ ]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    if not (last_typ or last_idx): return 0,0
    return last_idx + 1, last_idx + 1

Now we can create and register openai responses api's `acollect_stream`:

In [ ]:
#| export
@delegates(_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = _acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='gemini', **kwargs)
    async for o in res: yield o

In [ ]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, acollect_stream=acollect_stream)
api_registry.register('gemini', **norm_funcs)

In [ ]:
api_registry.apis['gemini']

namespace(norm_tool_calls=<function __main__.norm_tool_calls(resp)>,
          norm_parts=<function __main__.norm_parts(resp)>,
          norm_finish=<function __main__.norm_finish(resp, tcs=None)>,
          norm_usage=<function __main__.norm_usage(resp)>,
          acollect_stream=<function __main__.acollect_stream(resp, *, model=None, vendor_name=None)>)

The following tests demonstrate completions with streaming and non-streaming paths

#### Text

In [ ]:
inp = [{"role": "user", "parts": [{"text": "Hi how are you?"}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=6, completion_tokens=26, total_tokens=274, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=242, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 274, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 242})`

</details>

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): print_stream(o)

I'm doing well, thank you for asking! How are you doing today? Is

 there anything I can help you with?

In [ ]:
o

I'm doing well, thank you for asking! How are you doing today? Is there anything I can help you with?

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=6, completion_tokens=26, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=86, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 118, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 86})`

</details>

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=6, completion_tokens=26, total_tokens=274, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=242, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 274, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 242}),
 Usage(prompt_tokens=6, completion_tokens=26, total_tokens=118, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=86, raw={'promptTokenCount': 6, 'candidatesTokenCount': 26, 'totalTokenCount': 118, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}], 'thoughtsTokenCount': 86}))

#### Thinking

- **Thinking always happens internally** for Gemini 3/2.5 models — token cost visible in `thoughtsTokenCount` in usage metadata
- **Thought text in response** only returned when `include_thoughts: True` is explicitly set in `generation_config.thinking_config` — defaults to `false`
- **`thoughtSignature`** is always returned regardless of `includeThoughts` — required for multi-turn context continuity. Missing signatures in function calling result in a 400 error; for text/chat, omitting them degrades reasoning quality
- **`thinking_level`** controls reasoning depth: `high` (default for Gemini 3), `medium`, `low`, `minimal` (Flash only)

In [ ]:
inp = [{"role": "user", "parts": [{"text": "What is 31231231 * 12312?"}]}]
eff = {"thinking_config": {"include_thoughts": True}}
resp = await gem_cli.models.generate_content(model=mn,contents=inp,generation_config=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<details><summary>Thinking</summary>

**My Thought Process on Multiplying Two Large Numbers**

Okay, so I'm presented with the multiplication of two fairly large numbers: $31,231,231$ and $12,312$. My initial thought is to break down $12,312$ into more manageable components, like powers of ten and their multiples. I immediately think of $10,000 + 2,000 + 300 + 10 + 2$. This allows me to distribute the multiplication, transforming the problem into a series of easier calculations.

First, I start calculating the terms:
*   $31,231,231 \times 10,000$ (easy: just shift the decimal)
*   $31,231,231 \times 2,000$
*   $31,231,231 \times 300$
*   $31,231,231 \times 10$
*   $31,231,231 \times 2$

I compute each of these products, and then I begin to sum them. I'm carefully tracking the intermediate sums, keeping an eye out for potential errors. I briefly consider patterns within the multiplication of these large numbers as a check to spot a mistake.

Let's use the standard long multiplication method with the two numbers.

*   $31,231,231 \times 2 = 62,462,462$
*   $31,231,231 \times 10 = 312,312,310$
*   $31,231,231 \times 300 = 9,369,369,300$
*   $31,231,231 \times 2000 = 62,462,462,000$
*   $31,231,231 \times 10000 = 312,312,310,000$

Now I add the results in a column. I check the partial results as I compute the running sum to catch errors.

Finally, I arrive at the answer: $384,518,916,072$.

As a final check, I perform a quick order-of-magnitude estimation:  $31,231,231$ is roughly $3 \times 10^7$ and $12,312$ is roughly $1 \times 10^4$. Multiplying these, I get $3 \times 10^{11}$, which is in the ballpark of my calculated answer, giving me a degree of confidence. Reaching back and rechecking the column multiplication is a good confirmation.

My final answer, and it checks out, is that $31,231,231 \times 12,312 = 384,518,916,072$.




</details>

31,231,231 * 12,312 = **384,518,916,072**

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=20, completion_tokens=36, total_tokens=1504, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=1448, raw={'promptTokenCount': 20, 'candidatesTokenCount': 36, 'totalTokenCount': 1504, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1448})`

</details>

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, generation_config=eff, stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): print_stream(o)

🧠

To find the product of 31,2

31,231 and 12,312, you can multiply them step by step:

$$

31,231,231 \times 12,312 = 384

,518,916,072$$

**Breakdown of the calculation:**
*   $31,2

...

462$

Summing these parts:
$312,312,310,000 +

 62,462,462,000 + 9,369,3

69,300 + 312,312,310 + 62,

462,462 = \mathbf{384,518,916,0

72}$

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=20, completion_tokens=36, total_tokens=1504, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=1448, raw={'promptTokenCount': 20, 'candidatesTokenCount': 36, 'totalTokenCount': 1504, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 1448}),
 Usage(prompt_tokens=20, completion_tokens=359, total_tokens=1138, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=759, raw={'promptTokenCount': 20, 'candidatesTokenCount': 359, 'totalTokenCount': 1138, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 20}], 'thoughtsTokenCount': 759}))

#### Tool call

In [ ]:
inp = [{"role": "user", "parts": [{"text": 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]
tools = [{"functionDeclarations": [sch['function']]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp



🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'b': 20, 'a': 10})


<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=77, completion_tokens=38, total_tokens=221, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=106, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 221, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 106})`

</details>

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=tools, stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): print_stream(o)

In [ ]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name); test_eq(tc1.arguments,tc2.arguments); test_eq(tc1.server,tc2.server)

In [ ]:
test_tcs(comp.tool_calls,o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].data.keys()), set(o.message.content[0].data.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=77, completion_tokens=38, total_tokens=221, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=106, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 221, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 106}),
 Usage(prompt_tokens=77, completion_tokens=38, total_tokens=193, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=78, raw={'promptTokenCount': 77, 'candidatesTokenCount': 38, 'totalTokenCount': 193, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 77}], 'thoughtsTokenCount': 78}))

#### Web search

Gemini's Google Search is a transparent server-side tool — unlike Anthropic/OpenAI, it produces no `functionCall` parts or tool_use blocks. Search results appear as `groundingMetadata` on the candidate, detected by `usage_from_gemini` to set `server_tool_use: {"google_search": 1}`. The response content is plain text only, so `tool_calls` is always `[]`.

In [ ]:
inp = [{"role": "user", "parts": [{"text": "What is the weather in Istanbul today?"}]}]
resp = await gem_cli.models.generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Today, Friday, April 24, 2026, the weather in Istanbul is primarily **sunny** and mild. 

Here are the key details for today's forecast:
*   **Temperature:** Current temperature is approximately **16°C (60°F)**. The daytime high is expected to reach **16°C (61°F)**, while the low tonight will drop to around **10°C (50°F)**.
*   **Conditions:** It is sunny throughout the day, with a few clouds potentially appearing later in the evening.
*   **Precipitation:** There is a very low chance of rain (around 10% during the day and 0% currently).
*   **Humidity:** Humidity levels are around **56% to 63%**, making for a relatively comfortable day.

If you are planning to be outdoors, it is a great day for it, though you might want a light jacket for the cooler evening hours.

<details>

- model: `models/gemini-3-flash-preview`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=73, completion_tokens=213, total_tokens=941, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=655, raw={'promptTokenCount': 73, 'candidatesTokenCount': 213, 'totalTokenCount': 941, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 73}], 'thoughtsTokenCount': 655, 'server_tool_use': {'google_search': 1}})`

</details>

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

Today in Istanbul, it is mostly cloudy with comfortable spring temperatures.

*   **Temperature:** The high is expected to reach approximately **17°C to 19°C (63°F to 66°F)**, while the low tonight will drop to around **9°C (48°F)**.
*   **Conditions:** The day will remain mostly cloudy, transitioning to clear skies by tonight.
*   **Precipitation:** There is a low chance of rain, currently estimated at about **10%**.
*   **Wind & Humidity:** Winds are light, coming from the northeast at around 11 km/h, and humidity is moderate at about 58%.

If you're heading out, a light jacket or sweater is recommended, especially for the cooler evening hours. Sunset is expected at 7:57 PM local time.

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=51, completion_tokens=185, total_tokens=573, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=337, raw={'promptTokenCount': 51, 'candidatesTokenCount': 185, 'totalTokenCount': 573, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 51}], 'thoughtsTokenCount': 337, 'server_tool_use': {'google_search': 1}})`

</details>

Citations are yielded in `_acollect_stream`, but unlike Anthropic Gemini yields it only in the last delta, so we can't format it in realtime

In [ ]:
test_eq(comp.finish_reason, o.finish_reason)

In [ ]:
comp.usage, o.usage

(Usage(prompt_tokens=73, completion_tokens=213, total_tokens=941, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=655, raw={'promptTokenCount': 73, 'candidatesTokenCount': 213, 'totalTokenCount': 941, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 73}], 'thoughtsTokenCount': 655, 'server_tool_use': {'google_search': 1}}),
 Usage(prompt_tokens=51, completion_tokens=185, total_tokens=573, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=337, raw={'promptTokenCount': 51, 'candidatesTokenCount': 185, 'totalTokenCount': 573, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 51}], 'thoughtsTokenCount': 337, 'server_tool_use': {'google_search': 1}}))

## Denormalize

Helpers to transform internal representation types back to API payload

### Msgs

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Part(type=PartType.text, text=txt)])

Helpers to translate back to provider specific inputs from our internal `Msg` representation

This is not exported -- media tool results are added later

In [ ]:
def denorm_tool_result(p:Part):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    d = p.data or {}
    fr = dict(name=d.get('name',''), response={"content": str(p.text)})
    if d.get('id'): fr['id'] = d['id']
    return dict(functionResponse=fr)

This is not exported -- media inputs are added later

In [ ]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = [dict(text=p.text or '') for p in m.content if p.type == PartType.text]
    return dict(role='user', parts=parts)

In [ ]:
#| export
def denorm_tool_use(p:Part):
    "Convert canonical tool_use Part to Gemini functionCall part."
    d = p.data or {}
    fc = dict(name=d.get('name',''), args=d.get('arguments') or {})
    if d.get('id'): fc['id'] = d['id']
    part = dict(functionCall=fc)
    part['thoughtSignature'] = d.get('thoughtSignature', 'skip_thought_signature_validator')
    return part

def denorm_assistant(m:Msg):
    "Convert canonical assistant Msg to Gemini model content."
    parts = []
    for p in m.content:
        if   p.type == PartType.thinking: parts.append(dict(text=p.text or '', thought=True))
        elif p.type == PartType.text:     parts.append(dict(text=p.text or ''))
        elif p.type == PartType.tool_use: parts.append(denorm_tool_use(p))
    return dict(role='model', parts=parts)

def denorm_tool(m:Msg):
    "Convert canonical tool Msg to Gemini user content with functionResponse parts."
    parts = [denorm_tool_result(p) for p in m.content if p.type == PartType.tool_result]
    return dict(role='user', parts=parts)

def denorm_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Gemini contents."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_user(m))
        elif m.role == 'assistant': res.append(denorm_assistant(m))
        elif m.role == 'tool':      res.append(denorm_tool(m))
    return res

### Tool Schemas

Gemini function calling parameters only support a subset of OpenAPI schema according to the [docs](https://ai.google.dev/gemini-api/docs/function-calling?example=meeting#function-declarations):

In [ ]:
#| export
_valid_gemini_sch = {'type', 'format', 'title', 'description', 'nullable', 'default',
    'items', 'minItems', 'maxItems', 'enum', 'properties', 'propertyOrdering',
    'required', 'minProperties', 'maxProperties', 'minimum', 'maximum',
    'minLength', 'maxLength', 'pattern', 'example', 'anyOf'}

def _gem_filter_sch(s):
    if isinstance(s, list): return [_gem_filter_sch(x) for x in s]
    if not isinstance(s, dict): return s
    return {k: _gem_filter_sch(v) for k,v in s.items() if k in _valid_gemini_sch}

def denorm_tool_schs(tools):
    "Convert canonical tools to Gemini format."
    fn_decls, other = [], []
    for t in tools:
        fn = fn_schema(t)
        if fn is None: other.append(t); continue
        name, desc, params = fn
        params['properties'] = {k:_gem_filter_sch(v) for k,v in params['properties'].items()}
        fn_decls.append(dict(name=name, description=desc, parameters=params))
    out = other[:]
    if fn_decls: out.insert(0, dict(functionDeclarations=fn_decls))
    return out

In [ ]:
def view_test(
    path:str, # Path to directory or file to view
    view_range:tuple[int,int]=None, # Optional 1-indexed (start, end) line range for files, end=-1 for EOF. Do NOT use unless it's known that the file is too big to keep in context—simply view the WHOLE file when possible
    nums:bool=False, # Whether to show line numbers
    skip_folders:tuple[str,...]=('_proc','__pycache__') # Folder names to skip when listing directories
):
    'Test tool'
    pass

In [ ]:
invalid_sch = lite_mk_func(view_test)
gem_func = denorm_tool_schs([invalid_sch])
test_eq('prefixItems' not in gem_func[0]['functionDeclarations'][0]['parameters']['properties']['view_range'], True)

In [ ]:
test_eq(denorm_tool_schs([sch]), [{'functionDeclarations': [{'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}]}])
test_eq(denorm_tool_schs([{"googleSearch": {}}]), [{"googleSearch": {}}])
test_eq(denorm_tool_schs([sch, {"googleSearch": {}}]), [{'functionDeclarations': [{'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}]}, {"googleSearch": {}}])

### Tool Choice

In [ ]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to Gemini toolConfig."
    if v is None: return None
    if v in ('auto',):                    return {'functionCallingConfig': {'mode': 'AUTO'}}
    if v in ('required', 'any', 'force'): return {'functionCallingConfig': {'mode': 'ANY'}}
    if v in ('none', 'off', 'disabled'):  return {'functionCallingConfig': {'mode': 'NONE'}}
    return {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': [v]}}

Modes

In [ ]:
test_eq(denorm_tool_choice('auto'), {'functionCallingConfig': {'mode': 'AUTO'}})
test_eq(denorm_tool_choice('none'), {'functionCallingConfig': {'mode': 'NONE'}})

Tool name

In [ ]:
test_eq(denorm_tool_choice('simple_add'), {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': ['simple_add']}})

None

In [ ]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

In [ ]:
#| export
_gem_think_budgets = dict(minimal=128, low=1024, medium=2048, high=4096)
_gem_think_levels  = dict(minimal='low', low='low', medium='medium', high='high')

def denorm_reasoning(v, model=''):
    "Map canonical reasoning_effort to Gemini thinkingConfig (uses thinkingLevel for Gemini 3+)."
    err = ValueError(f"Invalid reasoning effort for Gemini: {v}, accepted string values are: {list(_gem_think_budgets)} and dicts are passthrough")
    if v is None: return None
    elif isinstance(v, dict): return v
    elif isinstance(v, str) and v in _gem_think_budgets:
        # defaults to includeThoughts same as litellm
        if 'gemini-3' in model: return {'thinkingLevel': _gem_think_levels.get(v, 'medium'), 'includeThoughts': True}
        return {'thinkingBudget': _gem_think_budgets.get(str(v).lower(), 1024), 'includeThoughts': True}

In [ ]:
test_eq(denorm_reasoning('medium', 'models/gemini-2.5-flash'), {'thinkingBudget': 2048, 'includeThoughts': True})
test_eq(denorm_reasoning('low', 'models/gemini-3-flash-preview'), {'thinkingLevel': 'low', 'includeThoughts': True})
test_eq(denorm_reasoning('high', 'models/gemini-3-flash-preview'), {'thinkingLevel': 'high', 'includeThoughts': True})
test_eq(denorm_reasoning(None), None)

### Web Search Options

In [ ]:
#| export
def denorm_web_search(v): return {"googleSearch": {}}

### System

In [ ]:
#| export
def denorm_system(sp): return dict(parts=[{'text': sys_text(part_txt(sp))}])

In [ ]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

Gemini passes sp to `system_instruction`

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]), system_instruction=denorm_system(sp), stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

I be a salt-crusted buccaneer sailin' the seven seas in search o' gold and glory!

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=22, completion_tokens=24, total_tokens=328, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=282, raw={'promptTokenCount': 22, 'candidatesTokenCount': 24, 'totalTokenCount': 328, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 22}], 'thoughtsTokenCount': 282})`

</details>

### Media Inputs

`fastllm` canonicalizes multimodal inputs into `Part(type, text, data)` where `text` carries the URL or data URL, and `data` is reserved for optional metadata. Each provider's denorm layer converts this canonical form into the provider's wire format.

In [ ]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(denorm_image(p))
        elif p.type == PartType.input_audio: parts.append(denorm_audio(p))
        elif p.type == PartType.input_video: parts.append(denorm_video(p))
        elif p.type == PartType.input_file:  parts.append(denorm_file(p))
    return dict(role='user', parts=parts)

#### Image

In [ ]:
#| export
def denorm_image(p):
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "image/*"), "fileUri": p.text}}

Image URL

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

This image is a tranquil landscape photograph showing a view from a wooden deck across a calm lake towards a range of majestic, snow-capped mountains.

Key features of the image include:
*   **Foreground:** A dark brown wooden deck or platform with vertical planks.
*   **Middle ground:** A perfectly still lake that acts as a mirror, reflecting the trees and mountains.
*   **Background:** A shoreline lined with green trees and yellow grasses, and behind them, massive mountains covered in snow and ice. The scene is bathed in a bright, soft, blueish light, giving it a peaceful and misty atmosphere.

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=1086, completion_tokens=127, total_tokens=1767, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=554, raw={'promptTokenCount': 1086, 'candidatesTokenCount': 127, 'totalTokenCount': 1767, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}, {'modality': 'IMAGE', 'tokenCount': 1080}], 'thoughtsTokenCount': 554})`

</details>

Image data

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_b64), Part(type=PartType.text, text='What color is this pixel?')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

The pixel is red.

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=1096, completion_tokens=5, total_tokens=1223, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=122, raw={'promptTokenCount': 1096, 'candidatesTokenCount': 5, 'totalTokenCount': 1223, 'promptTokensDetails': [{'modality': 'IMAGE', 'tokenCount': 1089}, {'modality': 'TEXT', 'tokenCount': 7}], 'thoughtsTokenCount': 122})`

</details>

#### Audio

In [ ]:
#| export
def denorm_audio(p): 
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "audio/*"), "fileUri": p.text}}

In [ ]:
wav_data = httpx.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.input_audio, text=audio_b64), Part(type=PartType.text, text='What is this audio saying?')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

The audio is saying: "The sun rises in the east and sets in the west. This simple fact has been observed by humans for thousands of years."

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=181, completion_tokens=31, total_tokens=389, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=177, raw={'promptTokenCount': 181, 'candidatesTokenCount': 31, 'totalTokenCount': 389, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 7}, {'modality': 'AUDIO', 'tokenCount': 174}], 'thoughtsTokenCount': 177})`

</details>

#### Video

In [ ]:
#| export
def denorm_video(p):
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "video/mp4"), "fileUri": p.text}}

Only supported by Gemini models

In [ ]:
vid_url = "https://storage.googleapis.com/cloud-samples-data/video/animals.mp4"
msg1 = Msg(role='user', content=[Part(type=PartType.input_video, text=vid_url), Part(type=PartType.text, text='Concisely, what animals are in this video?')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

The animals featured in the video include:

* **Giraffes**
* **Tigers**
* **Elephants**
* **Otters**
* **Sloths** (both real and animated)
* **Deer** 
* **Rabbits** and **Foxes** (animated characters from *Zootopia*)

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=7483, completion_tokens=71, total_tokens=8004, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=450, raw={'promptTokenCount': 7483, 'candidatesTokenCount': 71, 'totalTokenCount': 8004, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 17}, {'modality': 'VIDEO', 'tokenCount': 7466}], 'thoughtsTokenCount': 450})`

</details>

In [ ]:
vid_yt = "https://www.youtube.com/watch?v=9hE5-98ZeCg"
msg1 = Msg(role='user', content=[Part(type=PartType.input_video, text=vid_yt), Part(type=PartType.text, text='Summarize this video in one sentence.')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

This video is a demonstration of Gemini 2.0's multimodal live streaming capabilities, showcasing its ability to interact with real-time screen content, define terms, handle interruptions, and recall previous parts of the conversation.

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=13575, completion_tokens=44, total_tokens=13888, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=269, raw={'promptTokenCount': 13575, 'candidatesTokenCount': 44, 'totalTokenCount': 13888, 'promptTokensDetails': [{'modality': 'VIDEO', 'tokenCount': 13566}, {'modality': 'TEXT', 'tokenCount': 9}], 'thoughtsTokenCount': 269})`

</details>

#### Files

OpenAI requires a filename, Anthropic requires the media type and Gemini requires the mime type

In [ ]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": url_mime(p.text, "application/pdf"), "fileUri": p.text}}

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_url), Part(type=PartType.text, text='What is the title of this paper?')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

The title of this paper is **"Attention Is All You Need"**.

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=7809, completion_tokens=15, total_tokens=7909, cached_tokens=6694, cache_creation_tokens=0, reasoning_tokens=85, raw={'promptTokenCount': 7809, 'candidatesTokenCount': 15, 'totalTokenCount': 7909, 'cachedContentTokenCount': 6694, 'promptTokensDetails': [{'modality': 'IMAGE', 'tokenCount': 7800}, {'modality': 'TEXT', 'tokenCount': 9}], 'cacheTokensDetails': [{'modality': 'TEXT', 'tokenCount': 7}, {'modality': 'IMAGE', 'tokenCount': 6687}], 'thoughtsTokenCount': 85})`

</details>

In [ ]:
pdf_b64 = f"data:application/pdf;base64,{base64.b64encode(httpx.get(pdf_url).content).decode()}"
msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_b64), Part(type=PartType.text, text='What is the title of this paper')])

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs([msg1]),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

The title of this paper is **"Attention Is All You Need."**

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=7808, completion_tokens=15, total_tokens=7939, cached_tokens=6694, cache_creation_tokens=0, reasoning_tokens=116, raw={'promptTokenCount': 7808, 'candidatesTokenCount': 15, 'totalTokenCount': 7939, 'cachedContentTokenCount': 6694, 'promptTokensDetails': [{'modality': 'IMAGE', 'tokenCount': 7800}, {'modality': 'TEXT', 'tokenCount': 8}], 'cacheTokensDetails': [{'modality': 'TEXT', 'tokenCount': 6}, {'modality': 'IMAGE', 'tokenCount': 6688}], 'thoughtsTokenCount': 116})`

</details>

### Media Tool Call Results

A tool call can return an `input_image` or `input_file`

In [ ]:
#| export
def denorm_tool_result(p:Part):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    d = p.data or {}
    fr = dict(name=d.get('name',''), response={"content": "" if isinstance(p.text, list) else str(p.text)})
    if d.get('id'): fr['id'] = d['id']
    if isinstance(p.text, list):
        parts = []
        for pp in p.text:
            if   pp.type == PartType.text:        parts.append({"text": pp.text or ""})
            elif pp.type == PartType.input_image: parts.append(denorm_image(pp))
            elif pp.type == PartType.input_file:  parts.append(denorm_file(pp))
            else: raise ValueError(f"Gemini tool_result does not support {pp.type}")
        fr['parts'] = parts
    return dict(functionResponse=fr)

In [ ]:
msgs = [Msg(role='user', content=[Part(type=PartType.text, text="What's in this image?")]),
        Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': '_test', 'name': 'test_img', 'arguments': {}, 'server': False})]),
        Msg(role='tool', content=[Part(type=PartType.tool_result, text=[Part(type=PartType.input_image, text=img_b64)], data={'id': '_test', 'name': 'test_img'})])]

In [ ]:
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_msgs(msgs),stream=True, _query={"alt": "sse"})
async for o in acollect_stream(resp): pass
o

This image consists of a solid, uniform field of bright red. There are no other colors, shapes, or objects present.

<details>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=1120, completion_tokens=25, total_tokens=1178, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=33, raw={'promptTokenCount': 1120, 'candidatesTokenCount': 25, 'totalTokenCount': 1178, 'promptTokensDetails': [{'modality': 'IMAGE', 'tokenCount': 1089}, {'modality': 'TEXT', 'tokenCount': 31}], 'thoughtsTokenCount': 33})`

</details>

## Payload

Function to create the payload:

In [ ]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(model=model, contents=denorm_msgs(msgs))
    if sp:=kwargs.get('system'): payload['system_instruction'] = denorm_system(sp)
    gen_config = {}
    if mt:=kwargs.get('max_tokens'):        gen_config['maxOutputTokens'] = mt
    if thk:=denorm_reasoning(kwargs.get('reasoning_effort'), model): gen_config['thinkingConfig'] = thk
    if (temp:=kwargs.get('temperature')) is not None: gen_config['temperature'] = temp
    if gen_config: payload['generation_config'] = gen_config
    gem_tools = denorm_tool_schs(kwargs.get('tools')) if kwargs.get('tools') else []
    if (wopts:=kwargs.get('web_search_options')) is not None: gem_tools.append(denorm_web_search(wopts))
    if gem_tools:
        payload['tools'] = gem_tools
        has_fn  = any('functionDeclarations' in t for t in gem_tools)
        has_srv = any(k in t for t in gem_tools for k in ('googleSearch','codeExecution','googleSearchRetrieval'))
        if has_fn and has_srv: payload.setdefault('tool_config', {})['includeServerSideToolInvocations'] = True
    if tchc:=denorm_tool_choice(kwargs.get('tool_choice')): payload.setdefault('tool_config', {}).update(tchc)
    if kwargs.get('stream'): payload.update(stream=True, _query={"alt": "sse"})
    return payload

Function to create the default headers:

In [ ]:
#| export
def get_hdrs(api_key=None):
    return {"x-goog-api-key": get_api_key(api_key, 'GEMINI_API_KEY')}

## Cost

A function to calculate api Completion cost from raw usage data, and model metadata returned by `get_model_info`

In [ ]:
#| export
def cost(usage, m):
    raw = usage.raw
    prompt_tot = raw.get('promptTokenCount', 0)
    tier = '_above_200k_tokens' if prompt_tot > 200_000 else ''
    in_rate    = m.get(f'input_cost_per_token{tier}')       or m.input_cost_per_token
    out_rate   = m.get(f'output_cost_per_token{tier}')      or m.output_cost_per_token
    cache_rate = m.get(f'cache_read_input_token_cost{tier}')or m.get('cache_read_input_token_cost', 0)
    audio_rate = m.get('input_cost_per_audio_token')  # None if not priced separately

    cached = raw.get('cachedContentTokenCount', 0)
    # Gemini 3 Pro supports bills audio at the standard input rate (no separate input_cost_per_audio_token key in the metadata)
    audio  = sum(d['tokenCount'] for d in raw.get('promptTokensDetails', []) if d.get('modality')=='AUDIO') if audio_rate else 0
    in_txt = prompt_tot - cached - audio

    thoughts = raw.get('thoughtsTokenCount', 0) or 0
    cands    = raw.get('candidatesTokenCount', 0) or 0
    reason_rate = m.get('output_cost_per_reasoning_token') or out_rate

    cost  = in_txt * in_rate + cands * out_rate
    cost += cached * cache_rate
    cost += audio  * (audio_rate or 0)
    cost += thoughts * reason_rate
    return cost

## Register API

Register the final API namespace:

- `norm_tool_calls`, `norm_parts`, `norm_finish`, `norm_usage` required during the Completion object construction.
- `mk_payload` is used to construct the api request payload
- `accollect_stream` is used in streaming api call path
- `get_hdrs` is used to create the deafult request headers
- `op_path` define the non-streaming and streaming OpenAPIClient op names

In [ ]:
#| export
api_ns = dict(norm_tool_calls=norm_tool_calls,
                norm_parts=norm_parts,
                norm_finish=norm_finish,
                norm_usage=norm_usage,
                acollect_stream=acollect_stream,
                mk_payload=mk_payload,
                cost=cost,
                get_hdrs=get_hdrs,
                op_path=('models.generate_content','models.stream_generate_content'))
api_registry.register('gemini', **api_ns)

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()